<a href="https://colab.research.google.com/github/eshghinezhad/DeepLearning/blob/master/LSTM%26GRU/9_1_LSTM%2C_GRU_v2026_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sequence Models

**This notebook contains material adapted from _Hands-On Machine Learning with Scikit-Learn and PyTorch_ by Aurelien Geron (specifically, from the notebook for Chapter 13).** Source materials may be found at [https://github.com/ageron/handson-mlp](https://github.com/ageron/handson-mlp).

# Setup

This project requires Python 3.10 or above:

In [ ]:
import sys

assert sys.version_info >= (3, 10)

Are we using Colab or Kaggle?

In [ ]:
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

If using Colab, the TorchMetrics library is not pre-installed so we must install it manually:

In [ ]:
if IS_COLAB:
    %pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 24.3 MB/s eta 0:00:00


We also need PyTorch ≥ 2.6.0:

In [ ]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

This chapter can be very slow without a hardware accelerator, so if we can find one, let's use it:

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

Let's issue a warning if there's no hardware accelerator available:

In [ ]:
if device == "cpu":
    print("Neural nets can be very slow without a hardware accelerator.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if IS_KAGGLE:
        print("Go to Settings > Accelerator and select GPU.")

As we did in earlier chapters, let's define the default font sizes to make the figures prettier:

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## Time Series data setup and utility functions

This long section (until the next heading) is simply repeating the setup from the last session (RNNs), so that it can be reused for other models.

In [ ]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def download_and_extract_ridership_data():
    tarball_path = Path("datasets/ridership.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/ridership.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets", filter="data")

download_and_extract_ridership_data()

Now let's clean up the data a bit:

In [ ]:
import pandas as pd
from pathlib import Path

path = Path("datasets/ridership/CTA_-_Ridership_-_Daily_Boarding_Totals.csv")
df = pd.read_csv(path, parse_dates=["service_date"])


In [ ]:
df.head()

,service_date,day_type,bus,rail_boardings,total_rides
0,2001-01-01,U,297192,126455,423647
1,2001-01-02,W,780827,501952,1282779
2,2001-01-03,W,824923,536432,1361355
3,2001-01-04,W,870021,550011,1420032
4,2001-01-05,W,890426,557917,1448343


In [ ]:
df.shape

(7701, 5)

In [ ]:
df.columns = ["date", "day_type", "bus", "rail", "total"]  # shorter names
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # no need for total, it's just bus + rail
df = df.drop_duplicates()  # remove duplicated months (2011-10 and 2014-07)

In [ ]:
df.head()

,day_type,bus,rail
date,,,
2001-01-01,U,297192,126455
2001-01-02,W,780827,501952
2001-01-03,W,824923,536432
2001-01-04,W,870021,550011
2001-01-05,W,890426,557917


In [ ]:
df.shape

(7639, 3)

In [ ]:
class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, series, window_length):
        self.series = series
        self.window_length = window_length

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length  # 1st index after window
        window = self.series[idx : end]
        target = self.series[end]
        return window, target

In [ ]:
from torch.utils.data import DataLoader
# torch.manual_seed(0)
# my_loader = DataLoader(my_dataset, batch_size=2, shuffle=True)
# for X, y in my_loader:
#     print("X:", X, " y:", y)

Before we continue looking at the data, let's split the time series into three periods, for training, validation and testing. We won't look at the test data for now:

In [ ]:
rail_train = torch.FloatTensor(df[["rail"]]["2016-01":"2018-12"].values / 1e6)
rail_valid = torch.FloatTensor(df[["rail"]]["2019-01":"2019-05"].values / 1e6)
rail_test = torch.FloatTensor(df[["rail"]]["2019-06":].values / 1e6)

In [ ]:
rail_train

tensor([[0.3198],
        [0.3655],
        [0.2877],
        ...,
        [0.3071],
        [0.2653],
        [0.3861]])

In [ ]:
window_length = 56
train_set = TimeSeriesDataset(rail_train, window_length)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_set = TimeSeriesDataset(rail_valid, window_length)
valid_loader = DataLoader(valid_set, batch_size=32)
test_set = TimeSeriesDataset(rail_test, window_length)
test_loader = DataLoader(test_set, batch_size=32)

In [ ]:
batch = next(iter(train_loader))

windows, targets = batch

print(len(windows))
print(len(windows[0]))
print(windows[0])
print(targets[0])

32
56
tensor([[0.7815],
        [0.7705],
        [0.7737],
        [0.7790],
        [0.4293],
        [0.3302],
        [0.7472],
        [0.7458],
        [0.6209],
        [0.2084],
        [0.4151],
        [0.3694],
        [0.2908],
        [0.7269],
        [0.7532],
        [0.7611],
        [0.7728],
        [0.7851],
        [0.4617],
        [0.3386],
        [0.7400],
        [0.7374],
        [0.7422],
        [0.7419],
        [0.7398],
        [0.4058],
        [0.2955],
        [0.7062],
        [0.7099],
        [0.7152],
        [0.7227],
        [0.7005],
        [0.4199],
        [0.2964],
        [0.6753],
        [0.6977],
        [0.6839],
        [0.6600],
        [0.5862],
        [0.3241],
        [0.2058],
        [0.1187],
        [0.3277],
        [0.4160],
        [0.4566],
        [0.4669],
        [0.2764],
        [0.2359],
        [0.1955],
        [0.5142],
        [0.5877],
        [0.5911],
        [0.5725],
        [0.2921],
        [0.2266],
    

In [ ]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=10, factor=0.1):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [ ]:
import torch.nn as nn
import torchmetrics

In [ ]:
# extra code – defines a utility function we'll reuse several time

def fit_and_evaluate(model, train_loader, valid_loader, lr, n_epochs=50,
                     patience=20, factor=0.1):
    loss_fn = nn.HuberLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95)
    metric = torchmetrics.MeanAbsoluteError().to(device)
    history = train(model, optimizer, loss_fn, metric,
                    train_loader, valid_loader, n_epochs=n_epochs,
                    patience=patience, factor=factor)
    return min(history["valid_metrics"]) * 1e6

In [ ]:
df_mulvar = df[["rail", "bus"]] / 1e6  # use both rail & bus series as input
df_mulvar

,rail,bus
date,,
2001-01-01,0.126455,0.297192
2001-01-02,0.501952,0.780827
2001-01-03,0.536432,0.824923
2001-01-04,0.550011,0.870021
2001-01-05,0.557917,0.890426
...,...,...
2021-11-26,0.189694,0.257700
2021-11-27,0.187065,0.237839
2021-11-28,0.147830,0.184817


In [ ]:
df_mulvar["next_day_type"] = df["day_type"].shift(-1)  # we know tomorrow's type
df_mulvar

,rail,bus,next_day_type
date,,,
2001-01-01,0.126455,0.297192,W
2001-01-02,0.501952,0.780827,W
2001-01-03,0.536432,0.824923,W
2001-01-04,0.550011,0.870021,W
2001-01-05,0.557917,0.890426,A
...,...,...,...
2021-11-26,0.189694,0.257700,A
2021-11-27,0.187065,0.237839,U
2021-11-28,0.147830,0.184817,W


In [ ]:
df_mulvar = pd.get_dummies(df_mulvar, dtype=float)  # one-hot encode day type
df_mulvar

,rail,bus,next_day_type_A,next_day_type_U,next_day_type_W
date,,,,,
2001-01-01,0.126455,0.297192,0.0,0.0,1.0
2001-01-02,0.501952,0.780827,0.0,0.0,1.0
2001-01-03,0.536432,0.824923,0.0,0.0,1.0
2001-01-04,0.550011,0.870021,0.0,0.0,1.0
2001-01-05,0.557917,0.890426,1.0,0.0,0.0
...,...,...,...,...,...
2021-11-26,0.189694,0.257700,1.0,0.0,0.0
2021-11-27,0.187065,0.237839,0.0,1.0,0.0
2021-11-28,0.147830,0.184817,0.0,0.0,1.0


In [ ]:
mulvar_train = torch.FloatTensor(df_mulvar["2016-01":"2018-12"].values)
mulvar_valid = torch.FloatTensor(df_mulvar["2019-01":"2019-05"].values)
mulvar_test = torch.FloatTensor(df_mulvar["2019-06":].values)

In [ ]:
class MulvarTimeSeriesDataset(TimeSeriesDataset):
    def __getitem__(self, idx):
        window, target = super().__getitem__(idx)
        return window, target[:1]

In [ ]:
window_length = 56
mulvar_train_set = MulvarTimeSeriesDataset(mulvar_train, window_length)
mulvar_train_loader = DataLoader(mulvar_train_set, batch_size=32, shuffle=True)
mulvar_valid_set = MulvarTimeSeriesDataset(mulvar_valid, window_length)
mulvar_valid_loader = DataLoader(mulvar_valid_set, batch_size=32)
mulvar_test_set = MulvarTimeSeriesDataset(mulvar_test, window_length)
mulvar_test_loader = DataLoader(mulvar_test_set, batch_size=32)

In [ ]:
class MultaskTimeSeriesDataset(TimeSeriesDataset):
    def __getitem__(self, idx):
        window, target = super().__getitem__(idx)
        return window, target[:2]

window_length = 56
multask_train_set = MultaskTimeSeriesDataset(mulvar_train, window_length)
multask_train_loader = DataLoader(multask_train_set, batch_size=32, shuffle=True)
multask_valid_set = MultaskTimeSeriesDataset(mulvar_valid, window_length)
multask_valid_loader = DataLoader(multask_valid_set, batch_size=32)
multask_test_set = MultaskTimeSeriesDataset(mulvar_test, window_length)
multask_test_loader = DataLoader(multask_test_set, batch_size=32)


In [ ]:
class ForecastAheadDataset(TimeSeriesDataset):
    def __len__(self):
        return len(self.series) - self.window_length - 14 + 1

    def __getitem__(self, idx):
        end = idx + self.window_length  # 1st index after window
        window = self.series[idx : end]
        target = self.series[end : end + 14, 0]  # 0 = rail ridership
        return window, target

In [ ]:
window_length = 56
ahead_train_set = ForecastAheadDataset(mulvar_train, window_length)
ahead_train_loader = DataLoader(ahead_train_set, batch_size=32, shuffle=True)
ahead_valid_set = ForecastAheadDataset(mulvar_valid, window_length)
ahead_valid_loader = DataLoader(ahead_valid_set, batch_size=32)
ahead_test_set = ForecastAheadDataset(mulvar_test, window_length)
ahead_test_loader = DataLoader(ahead_test_set, batch_size=32)

In [ ]:
batch_ahead_train = next(iter(ahead_train_loader))

windows_ahead, targets_ahead = batch_ahead_train

print(len(windows_ahead))
print(len(windows_ahead[0]))
print(windows_ahead[0])
print(targets_ahead[0])

32
56
tensor([[0.7632, 0.8738, 0.0000, 0.0000, 1.0000],
        [0.7325, 0.7553, 0.0000, 0.0000, 1.0000],
        [0.7715, 0.8223, 0.0000, 0.0000, 1.0000],
        [0.7416, 0.7654, 1.0000, 0.0000, 0.0000],
        [0.4524, 0.5082, 0.0000, 1.0000, 0.0000],
        [0.3363, 0.3884, 0.0000, 0.0000, 1.0000],
        [0.7084, 0.6956, 0.0000, 0.0000, 1.0000],
        [0.7201, 0.7363, 0.0000, 0.0000, 1.0000],
        [0.7579, 0.7549, 0.0000, 0.0000, 1.0000],
        [0.7457, 0.7237, 0.0000, 0.0000, 1.0000],
        [0.6760, 0.6925, 1.0000, 0.0000, 0.0000],
        [0.4540, 0.5069, 0.0000, 1.0000, 0.0000],
        [0.2935, 0.3397, 0.0000, 0.0000, 1.0000],
        [0.7384, 0.8094, 0.0000, 0.0000, 1.0000],
        [0.7721, 0.8482, 0.0000, 0.0000, 1.0000],
        [0.7573, 0.8112, 0.0000, 0.0000, 1.0000],
        [0.7581, 0.7971, 0.0000, 0.0000, 1.0000],
        [0.7539, 0.8194, 1.0000, 0.0000, 0.0000],
        [0.4675, 0.4937, 0.0000, 1.0000, 0.0000],
        [0.3308, 0.3735, 0.0000, 0.0000, 1.0

In [ ]:
class Seq2SeqDataset(ForecastAheadDataset):
    def __getitem__(self, idx):
        end = idx + self.window_length  # 1st index after window
        window = self.series[idx : end]
        target_period = self.series[idx + 1 : end + 14, 0]
        target = target_period.unfold(dimension=0, size=14, step=1)
        return window, target

In [ ]:
window_length = 56
seq_train_set = Seq2SeqDataset(mulvar_train, window_length)
seq_train_loader = DataLoader(seq_train_set, batch_size=32, shuffle=True)
seq_valid_set = Seq2SeqDataset(mulvar_valid, window_length)
seq_valid_loader = DataLoader(seq_valid_set, batch_size=32)
seq_test_set = Seq2SeqDataset(mulvar_test, window_length)
seq_test_loader = DataLoader(seq_test_set, batch_size=32)

# Deep RNNs with Layer Norm

In [ ]:
class SimpleRnnModelWithLN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.memory_cell = nn.Sequential(
            nn.Linear(input_size + hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.Tanh()
        )
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        batch_size, window_length, dimensionality = X.shape
        X_time_first = X.transpose(0, 1)
        H = torch.zeros(batch_size, self.hidden_size, device=X.device)
        for X_t in X_time_first:
            XH = torch.cat((X_t, H), dim=1)
            H = self.memory_cell(XH)
        return self.output(H)

In [ ]:
torch.manual_seed(42)
rnn_with_ln_model = SimpleRnnModelWithLN(input_size=5, hidden_size=32, output_size=14)
rnn_with_ln_model = rnn_with_ln_model.to(device)
fit_and_evaluate(rnn_with_ln_model, ahead_train_loader, ahead_valid_loader,
                 lr=0.05, n_epochs=2)

Epoch 1/2, train loss: 0.0693, train metric: 0.2941, valid metric: 0.1618
Epoch 2/2, train loss: 0.0207, train metric: 0.1647, valid metric: 0.1438


143809.512257576

# LSTMs

In [ ]:
class LstmModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, last_state = self.rnn(X)
        return self.output(outputs[:, -1])

In [ ]:
torch.manual_seed(42)
lstm_model = LstmModel(input_size=5, hidden_size=32, output_size=14)
lstm_model = lstm_model.to(device)
fit_and_evaluate(lstm_model, ahead_train_loader, ahead_valid_loader,
                 lr=0.05, n_epochs=2)

Epoch 1/2, train loss: 0.0940, train metric: 0.3618, valid metric: 0.2087
Epoch 2/2, train loss: 0.0258, train metric: 0.1747, valid metric: 0.1717


171688.54176998138

In [ ]:
class LstmModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.memory_cell = nn.LSTMCell(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        batch_size, window_length, dimensionality = X.shape
        X_time_first = X.transpose(0, 1)
        H = torch.zeros(batch_size, self.hidden_size, device=X.device)
        C = torch.zeros(batch_size, self.hidden_size, device=X.device)
        for X_t in X_time_first:
            H, C = self.memory_cell(X_t, (H, C))
        return self.output(H)

If you're using RNNCell, GRUCell, or LSTMCell, you are responsible for iterating over the time steps, so you often transpose the input to make the time dimension first.
If you're using RNN, GRU, or LSTM, PyTorch iterates over the time steps internally. With batch_first=True, you can simply pass the input as (batch_size, sequence_length, input_size) without any transpose.

In [ ]:
torch.manual_seed(42)
lstm_model = LstmModel(input_size=5, hidden_size=32, output_size=14)
lstm_model = lstm_model.to(device)
fit_and_evaluate(lstm_model, ahead_train_loader, ahead_valid_loader, lr=0.05, n_epochs=2)

Epoch 1/2, train loss: 0.0940, train metric: 0.3618, valid metric: 0.2087
Epoch 2/2, train loss: 0.0258, train metric: 0.1747, valid metric: 0.1717


171688.46726417542

# GRUs

In [ ]:
class GruModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, last_state = self.rnn(X)
        return self.output(outputs[:, -1])

In [ ]:
torch.manual_seed(42)
gru_model = GruModel(input_size=5, hidden_size=32, output_size=14)
gru_model = gru_model.to(device)
fit_and_evaluate(gru_model, ahead_train_loader, ahead_valid_loader,
                 lr=0.05, n_epochs=2)

Epoch 1/2, train loss: 0.0813, train metric: 0.3289, valid metric: 0.1561
Epoch 2/2, train loss: 0.0212, train metric: 0.1757, valid metric: 0.1410


141033.56003761292

In [ ]:
class GruModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.memory_cell = nn.GRUCell(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        batch_size, window_length, dimensionality = X.shape
        X_time_first = X.transpose(0, 1)
        H = torch.zeros(batch_size, self.hidden_size, device=X.device)
        for X_t in X_time_first:
            H = self.memory_cell(X_t, H)
        return self.output(H)

In [ ]:
torch.manual_seed(42)
gru_model = GruModel(input_size=5, hidden_size=32, output_size=14)
gru_model = gru_model.to(device)
fit_and_evaluate(gru_model, ahead_train_loader, ahead_valid_loader,
                 lr=0.05, n_epochs=2)

Epoch 1/2, train loss: 0.0813, train metric: 0.3289, valid metric: 0.1561
Epoch 2/2, train loss: 0.0212, train metric: 0.1757, valid metric: 0.1410


141033.3216190338

## Using One-Dimensional Convolutional Layers to Process Sequences

```
  |-----0-----||-----2----||-----4----||--...-||------52------||------54------|
         |-----1----||-----3----||-----5--...-51------||------53------|
X:  0  1  2  3  4  5  6  7  8  9 10 11 12 ...  104 105 106 107 108 109 110 111
Y:      from 4     6     8    10    12    ...      106     108     110     112
         to 17    19    21    23    25    ...      119     121     123     125
```

In [ ]:
class DownsamplingModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.conv = nn.Conv1d(input_size, hidden_size, kernel_size=4, stride=2) # (batch_size, channels, sequence_length)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        Z = X.permute(0, 2, 1)  # treat time as a spatial dimension (batch_size, sequence_length, features) -> (batch_size, channels, sequence_length)
        Z = self.conv(Z)
        Z = Z.permute(0, 2, 1)  # swap back time & features dimensions
        Z = torch.relu(Z)
        Z, _states = self.gru(Z)
        return self.linear(Z)

torch.manual_seed(42)
dseq_model = DownsamplingModel(input_size=1, hidden_size=32, output_size=14)
dseq_model = dseq_model.to(device)

In [ ]:
class DownsampledDataset(Seq2SeqDataset):
    def __getitem__(self, idx):
        window, target = super().__getitem__(idx)
        return window, target[3::2]  # crop the first 3 targets and downsample

window_length = 112
dseq_train_set = DownsampledDataset(rail_train, window_length)
dseq_train_loader = DataLoader(dseq_train_set, batch_size=32, shuffle=True)
dseq_valid_set = DownsampledDataset(rail_valid, window_length)
dseq_valid_loader = DataLoader(dseq_valid_set, batch_size=32)
dseq_test_set = DownsampledDataset(rail_test, window_length)
dseq_test_loader = DataLoader(dseq_test_set, batch_size=32)

In [ ]:
torch.manual_seed(42)
dseq_model = DownsamplingModel(input_size=1, hidden_size=32, output_size=14)
dseq_model = dseq_model.to(device)
fit_and_evaluate(dseq_model, dseq_train_loader, dseq_valid_loader,
                 lr=0.2, n_epochs=2)

Epoch 1/2, train loss: 0.0595, train metric: 0.2736, valid metric: 0.1971
Epoch 2/2, train loss: 0.0204, train metric: 0.1694, valid metric: 0.1572


157236.18865013123

## WaveNet

```
 ⋮
C2  /\ /\ /\ /\ /\ /\ /\ /\ /\ /\ /\ /\ /\...
   \  /  \  /  \  /  \  /  \  /  \  /  \     
     /    \      /    \      /    \          
C1  /\ /\ /\ /\ /\ /\ /\ /\ /\ /\ /\  /\ /...\
X: 0  1  2  3  4  5  6  7  8  9  10 11 12 ... 111
Y: 1  2  3  4  5  6  7  8  9  10 11 12 13 ... 112
 /14 15 16 17 18 19 20 21 22  23 24 25 26 ... 125
```

Here's a simple implementation of a causal convolution:

In [ ]:
import torch.nn.functional as F

class CausalConv1d(nn.Conv1d):
    def forward(self, X):
        padding = (self.kernel_size[0] - 1) * self.dilation[0]
        X = F.pad(X, (padding, 0))
        return super().forward(X)

Here's another common implementation, called the _chomp_ approach: we use regular padding (both left and right) then chop off the right part.

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1, **kwargs):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation # Dilation creates gaps between the values used by the kernel, while left padding supplies missing past positions and preserves sequence length.
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.pad, dilation=dilation, **kwargs) # Dilation controls the spacing between kernel elements.

    def forward(self, x):
        out = self.conv(x)
        return out[:, :, :-self.pad] if self.pad > 0 else out

In [ ]:
class WavenetModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        layers = []
        for dilation in (1, 2, 4, 8) * 2:
            conv = CausalConv1d(input_size, hidden_size, kernel_size=2,
                                dilation=dilation)
            layers += [conv, nn.ReLU()]
            input_size = hidden_size
        self.convs = nn.Sequential(*layers)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        Z = X.permute(0, 2, 1)
        Z = self.convs(Z)
        Z = Z.permute(0, 2, 1)
        return self.output(Z)

torch.manual_seed(42)
wavenet_model = WavenetModel(input_size=5, hidden_size=32, output_size=14)
wavenet_model = wavenet_model.to(device)

In [ ]:
fit_and_evaluate(wavenet_model, seq_train_loader, seq_valid_loader,
                 lr=0.1, n_epochs=150)

Epoch 1/150, train loss: 0.0847, train metric: 0.3390, valid metric: 0.1920
Epoch 2/150, train loss: 0.0191, train metric: 0.1731, valid metric: 0.1560
Epoch 3/150, train loss: 0.0167, train metric: 0.1527, valid metric: 0.1408
Epoch 4/150, train loss: 0.0162, train metric: 0.1594, valid metric: 0.1502
Epoch 5/150, train loss: 0.0162, train metric: 0.1577, valid metric: 0.1453
Epoch 6/150, train loss: 0.0161, train metric: 0.1576, valid metric: 0.1473
Epoch 7/150, train loss: 0.0162, train metric: 0.1580, valid metric: 0.1465
Epoch 8/150, train loss: 0.0162, train metric: 0.1583, valid metric: 0.1471
Epoch 9/150, train loss: 0.0161, train metric: 0.1577, valid metric: 0.1468
Epoch 10/150, train loss: 0.0161, train metric: 0.1577, valid metric: 0.1469
Epoch 11/150, train loss: 0.0162, train metric: 0.1586, valid metric: 0.1467
Epoch 12/150, train loss: 0.0162, train metric: 0.1570, valid metric: 0.1465
Epoch 13/150, train loss: 0.0161, train metric: 0.1585, valid metric: 0.1469
Epoch 14

140761.01779937744